[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch03-ml-workflow/02-bang-thong-va-kinh-te.ipynb)

# Tải ảnh lên hay suy luận tại chỗ: băng thông, tính toán và tiền

Chương 3 dùng một hệ thống sàng lọc bệnh võng mạc do đái tháo đường cho các phòng khám vùng xa, và
đặt hai bài tính trên khăn giấy vào đúng chỗ hệ thống phải chọn nơi đọc ảnh. Bài thứ nhất hỏi một
ngày ảnh của một phòng khám có tải lên kịp qua đường truyền của nó không. Bài thứ hai hỏi suy luận
(inference) trên cloud hay trên một thiết bị đặt tại phòng khám thì rẻ hơn, và sau bao lâu. Sổ tay
này tính lại cả hai, rồi đổi các giả định để xem kết luận đứng vững tới đâu.

**Bạn sẽ làm:**
1. tính lại bài toán băng thông (bandwidth) của chương cho một phòng khám;
2. quét các đòn bẩy mà chương nêu nhưng không so sánh: tải qua đêm, nén, đường truyền nhanh hơn,
   suy luận tại chỗ;
3. đếm tham số và phép tính của MobileNetV2 để kiểm hai con số của chương, rồi đo nó trên CPU này
   cạnh thời gian tải lên một ảnh;
4. đặt hai đường phục vụ vào định luật sắt (iron law), kể cả phần phụ trội cố định (fixed overhead);
5. tính lại kinh tế học cloud so với biên, rồi quét quy mô phòng khám để xem khi nào phương án biên
   còn hoàn vốn.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import pandas as pd
import torchvision

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a5cc2"]  # thứ tự màu cố định, như các sổ tay khác


def thoi_gian(f, lap=5):
    """Trung vị thời gian của f(), sau một lần chạy khởi động."""
    f()
    do = []
    for _ in range(lap):
        t0 = time.perf_counter()
        f()
        do.append(time.perf_counter() - t0)
    return float(np.median(do))


print("sẵn sàng")

## 1. Một ngày ảnh qua một đường lên 2 Mb/s

Bài tính thứ hai của chương: một phòng khám khám 150 bệnh nhân mỗi ngày, chụp 10 ảnh mỗi người, mỗi
ảnh 5 MB. Đường lên của nó là 2 Mb/s, tức megabit mỗi giây, nên phải chia cho 8 để ra megabyte.

**Dự đoán:** tải hết một ngày ảnh mất bao nhiêu giờ? Có vừa trong một ca làm việc 8 giờ không?

In [ ]:
BENH_NHAN = sach(150, nguon="150 bệnh nhân × 10 ảnh × 5 MB")
ANH_MOI_NGUOI = sach(10, nguon="150 bệnh nhân × 10 ảnh × 5 MB")
MB_MOI_ANH = sach(5, nguon="150 bệnh nhân × 10 ảnh × 5 MB")
MBIT_S = sach(2, nguon="2 Mb/s = 0,25 MB/s")
GIO_CA = sach(8, nguon="104,2 % của cửa sổ 8 giờ")

mb_ngay = BENH_NHAN * ANH_MOI_NGUOI * MB_MOI_ANH
theo_sach("dữ liệu một ngày (MB)", mb_ngay, sach=7500, nguon="= 7.500 MB (7,5 GB)/ngày")
theo_sach("dữ liệu một ngày (GB)", mb_ngay / 1000, sach=7.5, nguon="(7,5 GB)/ngày")
MB_S = MBIT_S / 8                     # 8 bit một byte
theo_sach("đường lên (MB/s)", MB_S, sach=0.25, nguon="2 Mb/s = 0,25 MB/s")
giay = mb_ngay / MB_S
theo_sach("thời gian tải (giây)", giay, sach=30000, nguon="30.000 giây")
theo_sach("thời gian tải (giờ)", giay / 3600, sach=8.3, nguon="8,3 giờ")
theo_sach("phần của ca 8 giờ (%)", 100 * giay / 3600 / GIO_CA, sach=104.2, nguon="104,2 % của cửa sổ 8 giờ")

TOM_TAT_KB = sach(10, nguon="10 KB/bệnh nhân")
kb_tho_moi_nguoi = ANH_MOI_NGUOI * MB_MOI_ANH * 1000
theo_sach("ảnh thô so với bản tóm tắt", kb_tho_moi_nguoi / TOM_TAT_KB, sach=5000,
          nguon="10 KB/bệnh nhân · ~5.000×", tol=1)
vua_ca = GIO_CA * 3600 * MB_S / (ANH_MOI_NGUOI * MB_MOI_ANH)
print(f"một ca {GIO_CA} giờ tải được ảnh thô của {vua_ca:.0f} bệnh nhân; phòng khám có {BENH_NHAN}")

Phép chia cho 8 là chỗ dễ trượt nhất: đọc 2 Mb/s thành 2 MB/s thì thời gian tải giảm 8 lần và bài
toán biến mất. Tính đúng, một ca 8 giờ chỉ chở được ảnh của 144 bệnh nhân, nên phòng khám 150 bệnh
nhân đã vượt, và đó là khi đường truyền không phải chở gì khác. Chương cũng nói rõ điều này chỉ đúng
nếu phòng khám **cần tải lên ngay trong ca**. Bản tóm tắt kết quả 10 KB thay cho 50 MB ảnh thô mỗi
người thì giảm khoảng 5,000 lần.

## 2. Các đòn bẩy chương nêu mà không so sánh

Chương liệt kê những phương án phép tính trên không so: gom theo lô qua đêm, nén, xếp lịch, hay một
đường truyền nhanh hơn. Hàm dưới đây tính phần cửa sổ bị chiếm cho một cấu hình bất kỳ, rồi quét số
bệnh nhân mỗi ngày. Ngoài hai cấu hình của chương, mọi con số khác ở đây là **giả định** của sổ tay
này, để thấy hình dạng của đánh đổi:

* tải qua đêm trong 16 giờ ngoài ca;
* nén 10 lần, mức chỉ nén có mất mát mới đạt (nén có mất mát làm đổi chính dữ liệu mô hình nhìn thấy,
  nên phải kiểm định lại);
* một đường truyền 10 Mb/s;
* một phương án lai: suy luận tại chỗ cho mọi người, nhưng vẫn tải ảnh thô của những ca được chuyển
  tuyến, vì bác sĩ ở tuyến trên cần xem ảnh, ở đây là 30 % số bệnh nhân.

**Dự đoán:** với phòng khám 150 bệnh nhân, đòn bẩy nào đưa nó xuống dưới 100 %? Đòn bẩy nào còn
chỗ cho 500 bệnh nhân một ngày?

In [ ]:
def phan_cua_so(benh_nhan, mb_moi_nguoi, mbit_s, gio):
    """Phần trăm của cửa sổ tải mà dữ liệu một ngày chiếm."""
    return 100 * benh_nhan * mb_moi_nguoi / (mbit_s / 8) / (gio * 3600)


TY_LE_CHUYEN = 0.30                     # giả định: phần bệnh nhân được chuyển tuyến
MB_TOM_TAT = TOM_TAT_KB / 1000
DON_BAY = {
    "ảnh thô, trong ca 8 giờ (sách)": (ANH_MOI_NGUOI * MB_MOI_ANH, MBIT_S, GIO_CA),
    "ảnh thô, qua đêm 16 giờ": (ANH_MOI_NGUOI * MB_MOI_ANH, MBIT_S, 16),
    "nén 10 lần, trong ca": (ANH_MOI_NGUOI * MB_MOI_ANH / 10, MBIT_S, GIO_CA),
    "đường truyền 10 Mb/s, trong ca": (ANH_MOI_NGUOI * MB_MOI_ANH, 10, GIO_CA),
    "tại chỗ, tải ảnh ca chuyển tuyến": (MB_TOM_TAT + TY_LE_CHUYEN * ANH_MOI_NGUOI * MB_MOI_ANH, MBIT_S, GIO_CA),
    "tại chỗ, chỉ tải tóm tắt 10 KB (sách)": (MB_TOM_TAT, MBIT_S, GIO_CA),
}
bang = []
for ten, (mb, mbit, gio) in DON_BAY.items():
    toi_da = gio * 3600 * mbit / 8 / mb             # số bệnh nhân lấp đúng 100 % cửa sổ
    bang.append({"cấu hình": ten, "% cửa sổ ở 150 bệnh nhân": phan_cua_so(BENH_NHAN, mb, mbit, gio),
                 "số bệnh nhân tối đa mỗi ngày": toi_da})
assert abs(bang[0]["% cửa sổ ở 150 bệnh nhân"] - 100 * giay / 3600 / GIO_CA) < 1e-9
pd.DataFrame(bang).round(2)

In [ ]:
x = np.arange(10, 1001, 10)
fig, ax = plt.subplots(figsize=(9, 4.4))
for (ten, (mb, mbit, gio)), mau in zip(DON_BAY.items(), MAU + ["0.35"]):
    ax.semilogy(x, phan_cua_so(x, mb, mbit, gio), color=mau, lw=2, label=ten)
ax.axhline(100, color="0.4", lw=1, ls="--", label="100 %: lấp kín cửa sổ")
ax.plot([BENH_NHAN], [100 * giay / 3600 / GIO_CA], "o", color=MAU[0], ms=8, mec="white", mew=1.5)
ax.annotate(f"phòng khám của chương:\n{BENH_NHAN} bệnh nhân, 104.2 %", (BENH_NHAN, 104.2),
            xytext=(25, 450), textcoords="data", fontsize=8,
            arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_xlabel("số bệnh nhân mỗi ngày của một phòng khám")
ax.set_ylabel("% cửa sổ tải bị chiếm (thang log)")
ax.set_title("Một ngày dữ liệu chiếm bao nhiêu đường truyền")
ax.set_xlim(0, 1000)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout()
plt.show()

Đồ thị cho thấy hai loại đòn bẩy rất khác nhau:

* **Tải qua đêm, nén, đường truyền nhanh hơn** đều dời cả đường cong xuống, đúng bằng một hệ số. Chúng
  mua thêm một bội số bệnh nhân, nhưng ảnh thô vẫn đi qua đường truyền và vẫn tăng theo quy mô. Tải
  qua đêm còn đổi một thứ khác: nếu ảnh được đọc trên cloud, kết quả đến muộn một ngày, nên cách này chỉ
  hợp khi không cần trả kết quả ngay trong buổi khám.
* **Suy luận tại chỗ** đổi bản chất của thứ đi qua đường truyền, từ ảnh sang kết quả, nên đường của nó
  nằm thấp hơn 5,000 lần. Phương án lai cho thấy cái giá thật nằm ở phần ảnh vẫn phải gửi đi: khi ảnh
  của ca chuyển tuyến vẫn phải lên tuyến trên, mức giảm chỉ còn khoảng 1 chia cho tỉ lệ chuyển tuyến,
  ở đây hơn 3 lần.

Đó là lý do chương nói ràng buộc băng thông biến hạ tầng thành một quyết định của bước thu thập dữ
liệu: chọn phương án nào thì phải biết từ trước mô hình chạy ở đâu và cần những ảnh nào.

## 3. Đổi băng thông lấy tính toán: MobileNetV2 trên CPU này

Suy luận tại chỗ không miễn phí: nó chuyển gánh nặng từ đường truyền sang phần cứng ở phòng khám.
Chương lấy MobileNetV2 làm ví dụ một mô hình thị giác cho thiết bị di động, với ràng buộc khoảng
14 MB và khoảng 600 MFLOP. torchvision có sẵn kiến trúc này; dựng nó với trọng số ngẫu nhiên là đủ để
đếm và đo, vì số tham số và số phép tính không phụ thuộc giá trị trọng số.

Một phép nhân và một phép cộng là một MAC, tức 2 FLOP. Mỗi tầng tích chập có
$H_\text{ra} \cdot W_\text{ra} \cdot C_\text{ra} \cdot (C_\text{vào}/g) \cdot k^2$ MAC, với $g$ là số
nhóm: tích chập tách theo chiều sâu có $g = C_\text{vào}$, nên nó rẻ hơn hẳn.

In [ ]:
dat_hat_giong()
mobilenet = torchvision.models.mobilenet_v2(weights=None).eval()
so_tham_so = sum(p.numel() for p in mobilenet.parameters())
theo_sach("MobileNetV2, FP32 (MB)", so_tham_so * 4 / 1e6, sach=14, nguon="14 MB · 600 MFLOP", tol=0.5)

mac = {"tích chập 1×1": 0, "tích chập 3×3 theo chiều sâu": 0, "tích chập 3×3 thường": 0, "tuyến tính": 0}
mac_neu_thuong = 0  # nếu mỗi tích chập theo chiều sâu là một tích chập 3×3 thường, cùng số kênh


def dem_mac(tang, vao, ra):
    global mac_neu_thuong
    if isinstance(tang, torch.nn.Linear):
        mac["tuyến tính"] += tang.in_features * tang.out_features * ra.shape[0]
        return
    k2 = tang.kernel_size[0] * tang.kernel_size[1]
    if tang.groups > 1:
        mac["tích chập 3×3 theo chiều sâu"] += ra.numel() * k2
        mac_neu_thuong += ra.numel() * tang.in_channels * k2
    else:
        mac["tích chập 1×1" if k2 == 1 else "tích chập 3×3 thường"] += ra.numel() * tang.in_channels * k2


moc = [t.register_forward_hook(dem_mac) for t in mobilenet.modules()
       if isinstance(t, (torch.nn.Conv2d, torch.nn.Linear))]
anh_224 = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    mobilenet(anh_224)
for m in moc:
    m.remove()
tong_mac = sum(mac.values())
theo_sach("MobileNetV2 ở 224×224 (MFLOP)", 2 * tong_mac / 1e6, sach=600, nguon="14 MB · 600 MFLOP", tol=10)
for kieu, n in mac.items():
    print(f"{kieu:>30}: {2 * n / 1e6:7.1f} MFLOP ({100 * n / tong_mac:4.1f} %)")
tong_neu_thuong = tong_mac - mac["tích chập 3×3 theo chiều sâu"] + mac_neu_thuong
print(f"nếu mọi tích chập theo chiều sâu là tích chập 3×3 thường: {2 * tong_neu_thuong / 1e9:.2f} GFLOP, "
      f"gấp {tong_neu_thuong / tong_mac:.1f} lần")
NGAN_SACH_MB = sach(500, nguon="< 500 MB · 4–8 GB · dưới 20 W")
print(f"{so_tham_so:,} tham số; artifact {so_tham_so * 4 / 1e6:.1f} MB so với ngân sách dưới {NGAN_SACH_MB} MB")

Cả hai con số của chương khớp. Bảng chia cho thấy tích chập theo chiều sâu chỉ chiếm một phần nhỏ số
phép, dù nó có mặt trong mọi khối của mạng: các tích chập 1×1 quanh nó gánh phần lớn công việc. Dòng
cuối cho thấy nó tiết kiệm được gì: thay nó bằng tích chập 3×3 thường cùng số kênh thì mạng nặng gấp
nhiều lần, vượt xa ngân sách 600 MFLOP. Đó là ý của chương khi nói MobileNetV2 vừa ngân sách nhờ tích
chập tách theo chiều sâu, không chỉ nhờ giảm số tham số.

**Dự đoán:** suy luận một ảnh trên CPU này mất bao lâu, so với 20 giây để tải lên **một** ảnh 5 MB
qua đường lên 0.25 MB/s?

In [ ]:
with torch.no_grad():
    giay_suy_luan = thoi_gian(lambda: mobilenet(anh_224), lap=10)
giay_tai_mot_anh = MB_MOI_ANH / MB_S
do_tren_may("MobileNetV2, một ảnh 224×224, batch 1", 1000 * giay_suy_luan, "ms")
print(f"tải lên một ảnh {MB_MOI_ANH} MB qua {MB_S} MB/s: {giay_tai_mot_anh:.0f} s")
do_tren_may("tải lên một ảnh so với suy luận nó tại chỗ", giay_tai_mot_anh / giay_suy_luan, "lần")
do_tren_may("thông lượng suy luận", 1 / giay_suy_luan, "ảnh/s")
do_tren_may(f"suy luận {BENH_NHAN * ANH_MOI_NGUOI:,} ảnh, một ngày của phòng khám", BENH_NHAN * ANH_MOI_NGUOI * giay_suy_luan, "s")

Trên bất kỳ CPU nào chạy được sổ tay này, suy luận một ảnh nhanh hơn việc đẩy ảnh đó qua đường lên
của phòng khám rất nhiều lần. CPU ở đây chỉ đứng thay cho thiết bị biên, như NVIDIA Jetson mà tình
huống của chương đánh giá; ảnh đáy mắt thật cũng lớn hơn 224×224 và phải được thu nhỏ trước. Nhưng bất
đẳng thức đủ rộng để những khác biệt đó không đảo được nó. Một phòng khám 150 bệnh nhân với 10 ảnh mỗi
người cần 1,500 lượt suy luận một ngày, việc mà chính CPU này làm xong trong ít phút, trong khi tải
chúng lên mất hơn 8 giờ.

## 4. Hai đường phục vụ trong định luật sắt

Định luật sắt của chương chia thời gian của một yêu cầu thành ba số hạng:

$$T = \frac{D_\text{vol}}{\text{BW}} + \frac{O}{R_\text{peak} \cdot \eta_\text{hw}} + L_\text{lat}.$$

Với **đường cloud** trên chính đường lên 2 Mb/s, số hạng dữ liệu của một ảnh 5 MB là 20 giây, và hai số
hạng còn lại không đáng kể bên cạnh nó. Bài tính kinh tế của chương ghi đường cloud mất **trên 200 ms**,
tức là nó giả định một đường truyền khác hẳn:

In [ ]:
MS_CLOUD = sach(200, nguon="trên 200 ms")
mb_s_can = MB_MOI_ANH / (MS_CLOUD / 1000)
print(f"để riêng số hạng dữ liệu của một ảnh {MB_MOI_ANH} MB còn {MS_CLOUD} ms: "
      f"{mb_s_can:.0f} MB/s = {8 * mb_s_can:.0f} Mb/s, gấp {mb_s_can / MB_S:.0f} lần đường lên của bài trước")

Hai bài tính của chương là hai kịch bản riêng, nên không có mâu thuẫn trong từng bài. Nhưng đặt
cạnh nhau, chúng cho thấy con số "trên 200 ms" chỉ đúng với một kết nối nhanh gấp cả trăm lần đường
lên của phòng khám vùng xa ở bài trước. Ở đúng phòng khám ấy, riêng việc tải lên đã mất khoảng 20 giây mỗi
ảnh.

Với **đường biên**, số hạng dữ liệu gần như bằng 0 vì ảnh đã nằm sẵn tại chỗ, số hạng tính toán là thời
gian vừa đo ở phần 3. Còn số hạng thứ ba thì sao? Bài tự kiểm tra cuối mục triển khai của chương cho
một ví dụ: suy luận trên thiết bị 15 ms, tra thông tin bệnh nhân trên cloud 60 ms, định dạng kết quả
cho hệ thống thông tin bệnh viện 40 ms, trong khi ngân sách là 100 ms.

**Dự đoán:** nếu thay mô hình bằng một mô hình nhanh gấp đôi, hệ thống có lọt ngân sách không?

In [ ]:
SUY_LUAN_MS = sach(15, nguon="15 + 60 + 40 = 115 ms")
TRA_CUU_MS = sach(60, nguon="15 + 60 + 40 = 115 ms")
DINH_DANG_MS = sach(40, nguon="15 + 60 + 40 = 115 ms")
NGAN_SACH_MS = sach(100, nguon="dưới 20 W · 100 ms")
theo_sach("tổng độ trễ (ms)", SUY_LUAN_MS + TRA_CUU_MS + DINH_DANG_MS, sach=115, nguon="15 + 60 + 40 = 115 ms")

phuong_an = {
    "như bài tự kiểm tra": (SUY_LUAN_MS, TRA_CUU_MS, DINH_DANG_MS),
    "mô hình nhanh gấp đôi": (SUY_LUAN_MS / 2, TRA_CUU_MS, DINH_DANG_MS),
    "mô hình tốn 0 ms": (0, TRA_CUU_MS, DINH_DANG_MS),
    "lưu đệm thông tin bệnh nhân tại chỗ": (SUY_LUAN_MS, 0, DINH_DANG_MS),
}
for ten, phan in phuong_an.items():
    du = NGAN_SACH_MS - sum(phan)
    ket = f"còn dư {du:.1f} ms" if du > 0 else f"vượt {-du:.1f} ms" if du < 0 else "dùng hết, không dư ms nào"
    print(f"{ten:>36}: {sum(phan):5.1f} ms, so với ngân sách {NGAN_SACH_MS} ms: {ket}")

fig, ax = plt.subplots(figsize=(9, 3.0))
nhan = ["suy luận (số hạng tính toán)", "tra cứu trên cloud (phụ trội)", "định dạng cho bệnh viện (phụ trội)"]
for j, (ten, phan) in enumerate(phuong_an.items()):
    trai = 0
    for i, gt in enumerate(phan):
        ax.barh(j, gt, left=trai, color=MAU[i], height=0.6, edgecolor="white", lw=2, label=nhan[i] if j == 0 else None)
        trai += gt
ax.axvline(NGAN_SACH_MS, color="0.3", lw=1, ls="--")
ax.set_xlim(0, 135)
ax.text(NGAN_SACH_MS + 1.5, 2.5, f"ngân sách {NGAN_SACH_MS} ms", fontsize=8, color="0.3", va="center")
ax.set_yticks(range(len(phuong_an)), list(phuong_an))
ax.invert_yaxis()
ax.set_xlabel("ms mỗi yêu cầu")
ax.set_title("Độ trễ đầu-cuối của đường biên trong bài tự kiểm tra")
ax.legend(loc="upper center", bbox_to_anchor=(0.45, -0.3), ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

Mô hình chỉ chiếm 15 trong 115 ms. Làm nó nhanh gấp đôi vẫn vượt ngân sách. Kể cả một mô hình không tốn
chút thời gian nào cũng để lại 100 ms phụ trội, đúng bằng cả ngân sách, không còn chỗ cho việc gì khác.
Chỉ khi bỏ lần tra cứu qua mạng thì độ trễ mới tụt xuống còn 55 ms.
Đây là số hạng $L_\text{lat}$ của định luật sắt: phần không phụ thuộc mô hình, và cũng là phần mà một
nhóm chỉ tối ưu mô hình sẽ không bao giờ nhìn thấy.

## 5. Cloud hay biên: bài tính kinh tế của chương

Bài tính thứ ba của chương xét 500 phòng khám, mỗi phòng khám 50 bệnh nhân một ngày, mỗi người một ảnh
tính phí sau khi đã chọn lọc và kiểm chất lượng tại chỗ, chạy 365 ngày một năm. Lưu ý đây là một kịch
bản khác bài băng thông ở trên, nơi mỗi phòng khám có 150 bệnh nhân và 10 ảnh mỗi người.

In [ ]:
PHONG_KHAM = sach(500, nguon="500 phòng khám · 50 bệnh nhân/ngày")
BN_NGAY = sach(50, nguon="500 phòng khám · 50 bệnh nhân/ngày")
NGAY_NAM = sach(365, trich="365 days/year")
GIA_CLOUD = sach(0.01, nguon="$0,01 vs $0,001 mỗi ảnh")
GIA_BIEN = sach(0.001, nguon="$0,01 vs $0,001 mỗi ảnh")
KET_NOI = sach(45_000, nguon="$91.250 + ~$45.000", tol=1)       # kết nối và vận hành mạng, mỗi năm
THIET_BI = sach(500, nguon="$500/thiết bị")
BAO_TRI = sach(25_000, nguon="~$25.000 + ~$9.125", tol=1)       # bảo trì thiết bị biên, mỗi năm

anh_nam = PHONG_KHAM * BN_NGAY * NGAY_NAM
theo_sach("ảnh mỗi tháng", anh_nam / 12, sach=760_000, nguon="~760.000 ảnh/tháng", tol=5000)
theo_sach("cloud: suy luận mỗi năm ($)", anh_nam * GIA_CLOUD, sach=91_250, nguon="$91.250")
cloud_nam = anh_nam * GIA_CLOUD + KET_NOI
theo_sach("cloud: tổng mỗi năm ($)", cloud_nam, sach=136_250, nguon="~$136.250/năm", tol=1)
dau_tu = PHONG_KHAM * THIET_BI
theo_sach("biên: đầu tư một lần ($)", dau_tu, sach=250_000, nguon="$250.000 CapEx")
theo_sach("biên: suy luận mỗi năm ($)", anh_nam * GIA_BIEN, sach=9125, nguon="~$25.000 + ~$9.125", tol=1)
bien_nam = BAO_TRI + anh_nam * GIA_BIEN
theo_sach("biên: tổng mỗi năm ($)", bien_nam, sach=34_125, nguon="+ ~$34.125/năm", tol=1)
tiet_kiem = cloud_nam - bien_nam
theo_sach("tiết kiệm mỗi năm ($)", tiet_kiem, sach=102_125, nguon="~$102.125 tiết kiệm", tol=1)
theo_sach("thời gian hoàn vốn (năm)", dau_tu / tiet_kiem, sach=2.4, nguon="~2,4 năm hoàn vốn", tol=0.05)
print(f"{anh_nam:,} ảnh mỗi năm · hoàn vốn sau {dau_tu / tiet_kiem:.3f} năm = {12 * dau_tu / tiet_kiem:.1f} tháng")

Mọi con số của chương khớp, kể cả khoảng 760,000 ảnh mỗi tháng, vốn là 9,125,000 ảnh mỗi năm chia 12.
Hai khoản có dấu "khoảng" trong sách, \$45,000 cho kết nối và \$25,000 cho bảo trì, là phân bổ giả định
của kịch bản, không phải giá đo được. Chúng cũng là hai khoản quyết định kết quả nhiều nhất, như phần
dưới cho thấy.

**Dự đoán:** nếu mỗi phòng khám chỉ có 20 bệnh nhân một ngày, phương án biên còn hoàn vốn trong 5 năm
không? Còn 10 bệnh nhân?

Để quét theo quy mô, cần biết khoản kết nối \$45,000 lớn lên thế nào khi số ảnh tăng. Chương không nói.
Sổ tay này thử hai **giả định** ở hai đầu: khoản đó tỉ lệ với số ảnh tải lên, hoặc khoản đó cố định.

In [ ]:
def hoan_von(bn_ngay, ket_noi_ti_le=True):
    anh = PHONG_KHAM * bn_ngay * NGAY_NAM
    ket_noi = KET_NOI * (bn_ngay / BN_NGAY if ket_noi_ti_le else 1)
    tk = (anh * GIA_CLOUD + ket_noi) - (BAO_TRI + anh * GIA_BIEN)
    return np.where(tk > 0, dau_tu / np.maximum(tk, 1e-9), np.inf)


assert abs(float(hoan_von(BN_NGAY)) - dau_tu / tiet_kiem) < 1e-9
# Kết nối tỉ lệ: tiết kiệm mỗi năm = bn · (500 · 365 · (0.01 − 0.001) + 45,000 / 50) − 25,000.
# Dưới ngưỡng bn này, tiết kiệm âm và phương án biên không bao giờ hoàn vốn.
nguong_bn = BAO_TRI / (PHONG_KHAM * NGAY_NAM * (GIA_CLOUD - GIA_BIEN) + KET_NOI / BN_NGAY)
print(f"ngưỡng không hoàn vốn (kết nối tỉ lệ): {nguong_bn:.1f} bệnh nhân/ngày")
for bn in (10, 20, 50, 150):
    print(f"{bn:>4} bệnh nhân/ngày: hoàn vốn {float(hoan_von(bn)):5.2f} năm nếu kết nối tỉ lệ với ảnh, "
          f"{float(hoan_von(bn, False)):5.2f} năm nếu kết nối cố định")
nam_thang = np.linspace(0, 6, 121)
bn_quet = np.arange(4, 201)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2))
ax1.plot(nam_thang, cloud_nam * nam_thang / 1000, color=MAU[0], lw=2, label="cloud")
ax1.plot(nam_thang, (dau_tu + bien_nam * nam_thang) / 1000, color=MAU[1], lw=2, label="biên")
t_hv = dau_tu / tiet_kiem
ax1.plot([t_hv], [cloud_nam * t_hv / 1000], "o", color="0.2", ms=7)
ax1.annotate(f"hoàn vốn: {t_hv:.1f} năm", (t_hv, cloud_nam * t_hv / 1000), xytext=(-95, 40),
             textcoords="offset points", fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax1.set_xlabel("năm")
ax1.set_ylabel("chi phí cộng dồn (nghìn $)")
ax1.set_title(f"{PHONG_KHAM} phòng khám, {BN_NGAY} bệnh nhân mỗi ngày")
ax1.legend(loc="upper left", fontsize=8)

ax2.plot(bn_quet, hoan_von(bn_quet), color=MAU[2], lw=2, label="kết nối tỉ lệ với số ảnh")
ax2.plot(bn_quet, hoan_von(bn_quet, False), color=MAU[3], lw=2, label="kết nối cố định \\$45,000/năm")
ax2.axvline(nguong_bn, color=MAU[2], ls=":", lw=1)
ax2.text(nguong_bn + 2, 0.4, f"dưới {nguong_bn:.1f} bệnh nhân: không bao giờ\nhoàn vốn (kết nối tỉ lệ)", fontsize=8, color="0.3")
ax2.plot([BN_NGAY], [t_hv], "o", color="0.2", ms=7)
ax2.annotate(f"kịch bản của chương: {BN_NGAY} bệnh nhân, {t_hv:.1f} năm", (BN_NGAY, t_hv), xytext=(20, 35),
             textcoords="offset points", fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax2.set_ylim(0, 10)
ax2.set_xlim(0, 200)
ax2.set_xlabel("số bệnh nhân mỗi ngày của một phòng khám")
ax2.set_ylabel("thời gian hoàn vốn (năm)")
ax2.set_title("Phương án biên hoàn vốn sau bao lâu")
ax2.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

Ở quy mô của chương, hai giả định cho cùng một điểm, vì chúng được chọn để khớp ở đó. Ra xa điểm đó thì
chúng tách nhau, và ở phòng khám nhỏ, kết luận đổi hẳn. Khi khoản kết nối tỉ lệ với số ảnh, có một
ngưỡng, khoảng 10 bệnh nhân một ngày, dưới đó tiền tiết kiệm mỗi năm không bù nổi \$25,000 bảo trì, và
phương án biên **không bao giờ** hoàn vốn. Khi khoản kết nối cố định, chính khoản đó trả cho thiết bị, nên phương án biên hoàn vốn cả
ở phòng khám rất nhỏ.

Vì thế chương mở đầu kết luận bằng "dưới những giả định của kịch bản này". Tiền chưa phải lý lẽ duy
nhất: theo chương, đường cloud trên 200 ms đã vượt mục tiêu 100 ms của tình huống, và mất kết nối là
dừng sàng lọc, còn phương án biên cho độ trễ dưới 50 ms và chạy được khi mất mạng. Đổi lại, phương án
biên đòi mô hình phải vừa bộ nhớ của thiết bị biên, và một quy trình cập nhật phức tạp hơn. Chương nhắc
rằng mô hình triển khai chọn ở bước định nghĩa bài toán quyết định phương án biên có khả thi hay không.

## Thử thêm

1. Ở phần 2, đổi `TY_LE_CHUYEN` thành 0.1 rồi 0.5. Phương án lai còn chở được bao nhiêu bệnh nhân mỗi
   ngày? Đường của nó gần đường tóm tắt hay gần đường ảnh thô hơn?
2. Ở phần 5, đổi `GIA_CLOUD` thành 0.005 và chạy lại hai ô cuối. Ở 50 bệnh nhân một ngày, phương án
   biên hoàn vốn sau bao lâu, và ở số bệnh nhân nào thì nó không còn hoàn vốn?
3. Giả định thiết bị biên chỉ dùng được 3 năm. Với mỗi giả định về khoản kết nối, phòng khám cần ít
   nhất bao nhiêu bệnh nhân một ngày để phương án biên hoàn vốn trước khi phải thay thiết bị?

## Tóm lại

* Một ngày ảnh thô của phòng khám 150 bệnh nhân là 7,500 MB, cần 8.3 giờ qua đường lên 2 Mb/s, tức
  104.2 % ca 8 giờ; bản tóm tắt 10 KB mỗi người giảm khoảng 5,000 lần.
* Tải qua đêm, nén hay đường truyền nhanh hơn chỉ dời đường thẳng một hằng số; suy luận tại chỗ đổi thứ
  đi qua đường truyền, và cái giá thật nằm ở phần ảnh vẫn phải gửi đi.
* MobileNetV2 có khoảng 14 MB tham số và khoảng 600 MFLOP ở 224×224; suy luận một ảnh nhanh hơn rất
  nhiều so với tải ảnh đó qua đường lên của phòng khám.
* Trong bài tự kiểm tra 115 ms, mô hình chỉ chiếm 15 ms: phần phụ trội cố định, không phải mô hình,
  quyết định ngân sách độ trễ.
* Ở 500 phòng khám × 50 bệnh nhân, phương án biên tiết kiệm \$102,125 mỗi năm và hoàn vốn sau khoảng
  2.4 năm; ở phòng khám nhỏ, kết luận phụ thuộc hẳn vào giả định về khoản kết nối.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Quy trình hệ thống học máy](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch03-ml-workflow/), dựng từ chương
[*ML Workflow*](https://mlsysbook.ai/vol1/ml_workflow/ml_workflow.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.